# Multimodal SFT: Image Classification

Prepare Stanford Dogs image-classification data, fine-tune `gpt-4o-2024-08-06`, and inspect service-produced training and validation metrics.

| Setting | Value |
|---|---|
| Classes | 120 |
| Per-class selection | First 50 sorted image paths |
| Training | Positions 0-39: 4,800 rows |
| Validation | Positions 40-44: 600 rows |
| Unused provenance split | Positions 45-49: 600 rows |
| Model | `gpt-4o-2024-08-06` |
| Method | Supervised, 2 epochs, learning-rate multiplier 0.5, seed 42 |
| Training type | `GlobalStandard` |

The workflow uploads training and validation data, monitors the job, and downloads loss and token-accuracy metrics. The unused split is retained for provenance only; no deployment or inference is required.

Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Setup

Run from this cookbook directory after copying `.env.template` to `.env`, setting `FOUNDRY_PROJECT_ENDPOINT`, installing `../../requirements.lock`, and authenticating with `az login`.

Authentication uses `DefaultAzureCredential`. The OpenAI client is created with `AIProjectClient.get_openai_client()`.

Preparation cells make no Foundry calls. Execute the client, upload, and training cells to submit or resume a paid job. Creation calls disable SDK retries.

In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import json
import os
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import pandas as pd
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIConnectionError, APITimeoutError, InternalServerError, RateLimitError

START = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (START, *START.parents) if (candidate / "demo.yaml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError(f"Could not locate the cookbook root from {START}.")
load_dotenv(ROOT / ".env")

OUTPUTS = ROOT / "outputs" / "training-only"
OUTPUTS.mkdir(parents=True, exist_ok=True)
PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()

MODEL = "gpt-4o-2024-08-06"
TRAINING_TYPE = "GlobalStandard"
SEED = 42
EPOCHS = 2
LEARNING_RATE_MULTIPLIER = 0.5
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}

credential = None
project_client = None
client = None

print({
    "project_endpoint": PROJECT_ENDPOINT,
    "model": MODEL,
    "training_type": TRAINING_TYPE,
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate_multiplier": LEARNING_RATE_MULTIPLIER,
})

## 2. Reproduce the original dataset and split

This is the original demo's deterministic rule: clean the folder name into a breed label, sort by breed and image path, retain the first 50 images per breed, and assign positions 0-39/40-44/45-49 to training/validation/unused provenance.

In [ ]:
DOG_URL = "https://www.kaggle.com/api/v1/datasets/download/jessicali9530/stanford-dogs-dataset"
DOWNLOAD_ROOT = ROOT / "data" / "downloads" / "stanford-dogs"

def locate_images_root() -> Path | None:
    candidates = [
        path for path in DOWNLOAD_ROOT.rglob("Images")
        if path.is_dir() and path.parent.name.lower() == "images"
    ]
    return candidates[0] if candidates else None

def get_images_root() -> Path:
    existing = locate_images_root()
    if existing is not None:
        return existing
    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    archive = DOWNLOAD_ROOT / "stanford-dogs-dataset.zip"
    urllib.request.urlretrieve(DOG_URL, archive)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(DOWNLOAD_ROOT / "stanford_dogs_dataset")
    archive.unlink(missing_ok=True)
    extracted = locate_images_root()
    if extracted is None:
        raise FileNotFoundError("Stanford Dogs Images directory was not found after extraction.")
    return extracted

def clean_breed(folder_name: str) -> str:
    return folder_name.split("-", 1)[1].replace("_", " ").title()

images_root = get_images_root()
records: list[dict[str, Any]] = []
for breed_dir in images_root.iterdir():
    if not breed_dir.is_dir():
        continue
    breed = clean_breed(breed_dir.name)
    for image_path in breed_dir.iterdir():
        if image_path.is_file():
            records.append({"breed": breed, "image_path": image_path})

dataset = pd.DataFrame(records).sort_values(["breed", "image_path"]).reset_index(drop=True)
selected = dataset.groupby("breed", sort=False, as_index=False).head(50).copy()
selected["position"] = selected.groupby("breed", sort=False).cumcount()
selected["split"] = selected["position"].map(
    lambda value: "train" if value < 40 else "validation" if value < 45 else "unused"
)
selected["filename"] = selected["image_path"].map(lambda value: value.name)
selected["image_id"] = selected.apply(
    lambda row: f"{row['breed']}::{row['position']:02d}::{row['filename']}", axis=1
)

counts = selected.groupby("split").size().to_dict()
class_counts = selected.groupby(["split", "breed"]).size().unstack(fill_value=0)
assert selected["breed"].nunique() == 120
assert counts == {"train": 4800, "unused": 600, "validation": 600}
assert (class_counts.loc["train"] == 40).all()
assert (class_counts.loc["validation"] == 5).all()
assert (class_counts.loc["unused"] == 5).all()
assert not selected.duplicated("image_path").any()

membership_path = OUTPUTS / "split-membership.csv"
selected[["image_id", "breed", "filename", "position", "split"]].to_csv(
    membership_path, index=False
)
print({"classes": 120, "rows": counts, "membership": str(membership_path)})

## 3. Build training and validation JSONL

The message shape and prompt match the source demo. Each selected JPEG is embedded without resizing or recompression, and image detail remains `low`. The unused 600-row split is not serialized or uploaded.

In [ ]:
breeds = list(selected["breed"].drop_duplicates())
system_prompt = (
    "Classify the following input image into one of the following categories: "
    f"[{', '.join(breeds)}]."
)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_jsonl(split: str, path: Path) -> dict[str, Any]:
    rows = selected[selected["split"] == split]
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows.itertuples(index=False):
            image = base64.b64encode(Path(row.image_path).read_bytes()).decode("ascii")
            payload = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{image}",
                                    "detail": "low",
                                },
                            }
                        ],
                    },
                    {"role": "assistant", "content": row.breed},
                ]
            }
            stream.write(json.dumps(payload) + "\n")
    return {
        "path": str(path),
        "rows": len(rows),
        "bytes": path.stat().st_size,
        "sha256": sha256(path),
    }

artifacts = {
    "training": write_jsonl("train", OUTPUTS / "train_classification.jsonl"),
    "validation": write_jsonl("validation", OUTPUTS / "validation_classification.jsonl"),
}
(OUTPUTS / "dataset-manifest.json").write_text(
    json.dumps(artifacts, indent=2), encoding="utf-8"
)
print(json.dumps(artifacts, indent=2))

## 4. Upload hash-keyed files and submit once

Remote filenames include the local SHA-256 prefix; saved file IDs are reused on rerun. The local run-state fingerprint includes both hashes, model, training type, seed, and hyperparameters. A new job ID is persisted immediately after creation; rerunning the notebook retrieves that job instead of submitting another one.

In [ ]:
if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in .env before running training.")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project_client.get_openai_client(max_retries=0)


In [ ]:
def status_value(resource: Any) -> str:
    value = getattr(resource, "status", "")
    return str(getattr(value, "value", value)).lower()

def resource_id(resource: Any) -> str:
    value = getattr(resource, "id", None)
    if not value:
        raise RuntimeError(f"Resource has no id: {resource!r}")
    return str(value)

def service_read(operation, name: str) -> Any:
    for attempt in range(1, 7):
        try:
            return operation()
        except (APIConnectionError, APITimeoutError, InternalServerError, RateLimitError) as error:
            if attempt == 6:
                raise
            delay = min(2 ** attempt, 30)
            print({"operation": name, "retry": attempt, "delay_seconds": delay, "error": type(error).__name__})
            time.sleep(delay)
    raise AssertionError("Retry loop ended unexpectedly.")

def wait_for_file(file_id: str) -> Any:
    started = time.monotonic()
    while True:
        remote = service_read(lambda: client.files.retrieve(file_id), "retrieve file")
        status = status_value(remote)
        print({"file": file_id, "status": status})
        if status in {"processed", "error", "deleted", "failed", "cancelled", "expired"}:
            if status != "processed":
                raise RuntimeError(f"File {file_id} ended in {status}.")
            return remote
        if time.monotonic() - started >= 900:
            raise TimeoutError(f"Timed out processing file {file_id}.")
        time.sleep(10)

def upload_file(local_path: Path, digest: str) -> Any:
    remote_name = f"{local_path.stem}-{digest[:16]}.jsonl"
    print({"file": remote_name, "action": "upload"})
    with local_path.open("rb") as stream:
        created = client.files.create(
            file=(remote_name, stream, "application/jsonl"), purpose="fine-tune"
        )
    return wait_for_file(resource_id(created))

contract = {
    "model": MODEL,
    "training_type": TRAINING_TYPE,
    "seed": SEED,
    "n_epochs": EPOCHS,
    "learning_rate_multiplier": LEARNING_RATE_MULTIPLIER,
    "training_sha256": artifacts["training"]["sha256"],
    "validation_sha256": artifacts["validation"]["sha256"],
}
fingerprint = hashlib.sha256(
    json.dumps(contract, sort_keys=True).encode("utf-8")
).hexdigest()
state_path = OUTPUTS / "run-state.json"
file_state_path = OUTPUTS / "file-state.json"

job = None
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    if state.get("fingerprint") != fingerprint:
        raise RuntimeError(
            "run-state.json belongs to a different dataset or training contract; "
            "move it aside only when intentionally starting a fresh run."
        )
    if state.get("project_endpoint") != PROJECT_ENDPOINT:
        raise RuntimeError("Saved run state has a different or missing project endpoint; verify its provenance before reuse.")
    if state.get("submission_pending") and not state.get("job_id"):
        raise RuntimeError("Previous submission outcome is unknown; reconcile the remote job and record its ID before retrying. Do not clear pending state to resubmit.")
    job = service_read(lambda: client.fine_tuning.jobs.retrieve(state["job_id"]), "retrieve saved job")
    print({"job": state["job_id"], "action": "reuse-local-state"})
else:
    if file_state_path.exists():
        file_state = json.loads(file_state_path.read_text(encoding="utf-8"))
        if (file_state.get("fingerprint") != fingerprint
                or file_state.get("project_endpoint") != PROJECT_ENDPOINT):
            raise RuntimeError("file-state.json does not match the current dataset contract.")
    else:
        file_state = {"fingerprint": fingerprint, "project_endpoint": PROJECT_ENDPOINT}

    remote_files = {}
    for split in ("training", "validation"):
        key = f"{split}_file_id"
        if key in file_state:
            remote_files[split] = wait_for_file(file_state[key])
            print({"file": file_state[key], "action": "reuse-local-state"})
            continue
        local_path = Path(artifacts[split]["path"])
        remote_files[split] = upload_file(local_path, artifacts[split]["sha256"])
        file_state[key] = resource_id(remote_files[split])
        file_state_path.write_text(json.dumps(file_state, indent=2), encoding="utf-8")

    pending_state = {
        "fingerprint": fingerprint,
        "project_endpoint": PROJECT_ENDPOINT,
        "contract": contract,
        "training_file_id": resource_id(remote_files["training"]),
        "validation_file_id": resource_id(remote_files["validation"]),
        "submission_pending": True,
    }
    state_path.write_text(json.dumps(pending_state, indent=2), encoding="utf-8")
    job = client.with_options(max_retries=0).fine_tuning.jobs.create(
        model=MODEL,
        training_file=resource_id(remote_files["training"]),
        validation_file=resource_id(remote_files["validation"]),
        seed=SEED,
        suffix="stanford-dogs-vision-sft",
        method={
            "type": "supervised",
            "supervised": {
                "hyperparameters": {
                    "n_epochs": EPOCHS,
                    "learning_rate_multiplier": LEARNING_RATE_MULTIPLIER,
                }
            },
        },
        extra_body={"trainingType": TRAINING_TYPE},
    )
    state = {
        "fingerprint": fingerprint,
        "project_endpoint": PROJECT_ENDPOINT,
        "job_id": resource_id(job),
        "submission_pending": False,
        "contract": contract,
        "training_file_id": resource_id(remote_files["training"]),
        "validation_file_id": resource_id(remote_files["validation"]),
    }
    state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
    print({"job": state["job_id"], "action": "created"})


## 5. Monitor to terminal and download service metrics

The job is polled until `succeeded`, `failed`, or `cancelled`. The complete terminal job object is saved before any failure is raised. On success, every service result file is downloaded and loss/token-accuracy columns are summarized when present.

In [ ]:
if job is not None:
    job_id = resource_id(job)
    monitor_started = time.monotonic()
    monitor_timeout_seconds = 20 * 60 * 60
    poll_interval_seconds = 5 * 60
    while True:
        job = service_read(lambda: client.fine_tuning.jobs.retrieve(job_id), "retrieve fine-tuning job")
        job_status = status_value(job)
        elapsed_seconds = time.monotonic() - monitor_started
        print({"job": job_id, "status": job_status, "elapsed_minutes": round(elapsed_seconds / 60, 1)})
        if job_status in TERMINAL_STATUSES:
            break
        if elapsed_seconds >= monitor_timeout_seconds:
            raise TimeoutError(
                f"Fine-tuning job {job_id} remained {job_status} after "
                f"{monitor_timeout_seconds / 3600:g} hours."
            )
        time.sleep(poll_interval_seconds)

    job_payload = job.model_dump(mode="json") if hasattr(job, "model_dump") else vars(job)
    (OUTPUTS / "terminal-job.json").write_text(
        json.dumps(job_payload, indent=2, default=str), encoding="utf-8"
    )

    if job_status != "succeeded":
        raise RuntimeError(
            f"Fine-tuning job {job_id} ended in {job_status}: {getattr(job, 'error', None)}"
        )

    result_ids = [str(value) for value in (getattr(job, "result_files", None) or [])]
    metric_summaries: dict[str, dict[str, float | int]] = {}
    downloaded: list[str] = []
    for index, result_id in enumerate(result_ids, start=1):
        response = service_read(lambda: client.files.content(result_id), "download result file")
        content = getattr(response, "content", None)
        if content is None:
            content = getattr(response, "read", lambda: b"")()
        if isinstance(content, str):
            content = content.encode("utf-8")
        result_path = OUTPUTS / f"service-results-{index}.csv"
        result_path.write_bytes(content)
        downloaded.append(str(result_path))

        frame = pd.read_csv(result_path)
        metric_columns = [
            column for column in frame.columns
            if "loss" in column.lower() or "token_accuracy" in column.lower()
        ]
        for column in metric_columns:
            values = pd.to_numeric(frame[column], errors="coerce").dropna()
            if values.empty:
                continue
            metric_summaries[column] = {
                "count": int(values.size),
                "first": float(values.iloc[0]),
                "last": float(values.iloc[-1]),
                "minimum": float(values.min()),
                "maximum": float(values.max()),
            }

    summary = {
        "project_endpoint": PROJECT_ENDPOINT,
        "model": MODEL,
        "training_type": TRAINING_TYPE,
        "job_id": job_id,
        "status": job_status,
        "dataset": artifacts,
        "result_files": result_ids,
        "downloaded_results": downloaded,
        "metrics": metric_summaries,
    }
    (OUTPUTS / "training-summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))

In [ ]:
close = getattr(client, "close", None)
if callable(close):
    close()
close = getattr(project_client, "close", None)
if callable(close):
    close()
if credential is not None:
    credential.close()
